# 🇵🇭 Philippine Address Geocoding Pipeline — v1.0 (Production)

**Architecture:** `clean → generate query variants → Nominatim (sequential fallback) → fuzzy local dictionary → null`

| Stage | Description |
|-------|-------------|
| **1. Normalization** | PH-specific noise removal, concatenated-text splitting, alias expansion |
| **2. Query Generation** | Progressive degradation: specific → general, always includes PH suffix |
| **3. Nominatim Geocoding** | OSM API, rate-limited (1 req/s), stops at first match |
| **4. Fuzzy Local Fallback** | RapidFuzz against dim_location barangay/city/province dictionary |
| **5. Batch Execution** | Deduplication + in-memory cache + optional parallel dispatch |

**Inputs required:**
- `dim_location_20260316_v2.csv` — PSGC reference table (barangay/city/province/region)
- `ph_address_alias_extended_v4.csv` — alias/abbreviation replacement pairs
- `ph_area_dictionary_v2.json` — city alias map, province alias map, area inference rules
- `sample_address.xlsx` — input addresses (column 0 = address string)

**Outputs:**
- `combined_addresses_<suffix>.xlsx` — all rows with geocode result, source, and confidence
- `valid_addresses_<suffix>.xlsx` — matched rows in standardized admin columns
- `invalid_addresses_<suffix>.xlsx` — unmatched rows in standardized admin columns

## Cell 0 — Environment Check

In [55]:
import importlib, sys

REQUIRED = {
    'pandas':     'pandas',
    'numpy':      'numpy',
    'rapidfuzz':  'rapidfuzz',
    'requests':   'requests',
    'tqdm':       'tqdm',
    'openpyxl':   'openpyxl',
    'xlsxwriter': 'xlsxwriter',
    'diskcache':  'diskcache',   # optional — persistent disk cache
}

missing, optional_missing = [], []
OPTIONAL = {'diskcache'}

for label, pkg in REQUIRED.items():
    try:
        mod = importlib.import_module(pkg)
        print(f'  ✓  {label:<14} {getattr(mod, "__version__", "n/a")}')
    except ImportError:
        if pkg in OPTIONAL:
            print(f'  ○  {label:<14} OPTIONAL — disk cache disabled')
            optional_missing.append(pkg)
        else:
            print(f'  ✗  {label:<14} NOT FOUND')
            missing.append(pkg)

if missing:
    print(f'\n⚠  Run: pip install {" ".join(missing)}')
    raise ImportError(f'Missing required packages: {missing}')
else:
    print('\n✅  All required dependencies satisfied.')

  ✓  pandas         3.0.1
  ✓  numpy          2.4.3
  ✓  rapidfuzz      3.14.3
  ✓  requests       2.32.5
  ✓  tqdm           4.67.3
  ✓  openpyxl       3.1.5
  ✓  xlsxwriter     3.2.9
  ○  diskcache      OPTIONAL — disk cache disabled

✅  All required dependencies satisfied.


## Cell 1 — Configuration

In [56]:
from pathlib import Path

# ── File paths ────────────────────────────────────────────────────────────────
BASE_DIR       = Path('..') / '..' / 'data'
INPUT_FILE     = str(BASE_DIR / 'sample' / 'ps_address_v3.csv')
DIM_LOCATION   = str(BASE_DIR / 'mapping'    / 'dim_location_20260316_v2.csv')
ALIAS_FILE     = str(BASE_DIR / 'utils'      / 'ph_address_alias_extended_v4.csv')
AREA_DICT_FILE = str(BASE_DIR / 'dictionary' / 'ph_area_dictionary_v2.json')
OUT_VALID      = str(BASE_DIR / 'output'     / 'valid'   / 'valid_addresses.xlsx')
OUT_INVALID    = str(BASE_DIR / 'output'     / 'invalid' / 'invalid_addresses.xlsx')
OUT_COMBINED   = str(BASE_DIR / 'output'     / 'combined_addresses.xlsx')

input_paths: list[Path] = []   # fill for batch mode

# ── Nominatim settings ───────────────────────────────────────────────────────
NOMINATIM_URL       = 'https://nominatim.openstreetmap.org/search'
NOMINATIM_USER_AGENT = 'ph-geocoding-pipeline/1.0 (your@email.com)'
NOMINATIM_RATE_LIMIT = 1.05          # seconds between requests (>=1.0 per OSM ToS)
NOMINATIM_TIMEOUT    = 10            # seconds per request
NOMINATIM_COUNTRY    = 'Philippines' # appended as country hint

# ── Fuzzy thresholds ─────────────────────────────────────────────────────────
FUZZY_CITY_HIGH  = 88
FUZZY_CITY_LOW   = 80
FUZZY_PROV_HIGH  = 90
FUZZY_PROV_LOW   = 82
FUZZY_BRGY_HIGH  = 88

# ── Pipeline behaviour ───────────────────────────────────────────────────────
MAX_ROWS: int | None = None          # None = process all rows
ENABLE_NOMINATIM     = False         # False = skip to fuzzy fallback directly
ENABLE_DISK_CACHE    = True          # requires diskcache
MAX_QUERY_VARIANTS   = 5             # max Nominatim queries per address
PARALLEL_WORKERS     = 1             # >1 uses thread pool BUT still rate-limited

print('Configuration loaded.')

Configuration loaded.


## Cell 2 — Imports & Logging

In [57]:
import re, json, time, logging, unicodedata, hashlib
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache

import pandas as pd
import numpy as np
import requests
from rapidfuzz import fuzz, process as rf_process
from tqdm.notebook import tqdm

# Disk cache (optional)
try:
    diskcache = importlib.import_module('diskcache')
    cache_dir = globals().get('CACHE_DIR', BASE_DIR / 'cache')
    _disk_cache = diskcache.Cache(str(cache_dir)) if ENABLE_DISK_CACHE else None
    print('diskcache: enabled')
except ImportError:
    _disk_cache = None
    print('diskcache: disabled (in-memory cache only)')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('ph_geocoding')
log.info('Imports OK')

16:51:21  INFO      Imports OK


diskcache: disabled (in-memory cache only)


## Cell 3 — Reference Data Loader

In [58]:
def _strip_accents(text: str) -> str:
    """Remove diacritics (ñ → n, é → e, etc.)"""
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(text))
        if unicodedata.category(c) != 'Mn'
    )


def _read_csv(path: str) -> pd.DataFrame:
    """Encoding-safe CSV reader (tries UTF-8-sig, UTF-8, latin1, cp1252)."""
    for enc in ('utf-8-sig', 'utf-8', 'latin1', 'cp1252'):
        try:
            return pd.read_csv(path, encoding=enc)
        except (UnicodeDecodeError, Exception):
            continue
    raise ValueError(f'Cannot read: {path}')


def load_reference(
    dim_path: Path,
    alias_path: Path,
    area_dict_path: Path,
) -> dict:
    """
    Load and index all reference data. Returns a single config dict
    consumed by the rest of the pipeline.
    """
    # ── dim_location ─────────────────────────────────────────────────────────
    dim = _read_csv(str(dim_path))
    dim['_city_norm'] = dim['city_name'].apply(lambda x: _strip_accents(x).upper().strip())
    dim['_prov_norm'] = dim['province_name'].apply(lambda x: _strip_accents(x).upper().strip())
    dim['_brgy_norm'] = dim['barangay_name'].apply(lambda x: _strip_accents(x).upper().strip())
    log.info(f'dim_location: {len(dim):,} rows')

    # ── alias table ──────────────────────────────────────────────────────────
    alias_df = _read_csv(str(alias_path))
    log.info(f'alias: {len(alias_df):,} pairs')

    # ── area dictionary ───────────────────────────────────────────────────────
    with open(area_dict_path, encoding='utf-8-sig') as f:
        area_dict = json.load(f)
    log.info(f'area_dict v{area_dict.get("version", "?")} loaded')

    unique_cities = sorted(dim['_city_norm'].unique().tolist())
    unique_provs  = sorted(dim['_prov_norm'].unique().tolist())
    unique_brgys  = sorted(dim['_brgy_norm'].unique().tolist())

    # ── city_canonical: alias → canonical dim city key ───────────────────────
    city_canonical: dict[str, str] = {}
    for city in unique_cities:
        city_canonical[city] = city
        for prefix in ('CITY OF ', 'MUNICIPALITY OF ', 'MUNICIPAL CITY OF '):
            if city.startswith(prefix):
                city_canonical.setdefault(city[len(prefix):].strip(), city)
        for suffix in (' CITY', ' MUNICIPALITY'):
            if city.endswith(suffix):
                city_canonical.setdefault(city[:-len(suffix)].strip(), city)

    # Area dict city aliases
    for city_key, meta in area_dict.get('cities', {}).items():
        ck = _strip_accents(city_key).upper()
        canonical = city_canonical.get(ck, ck)
        for alias in meta.get('aliases', []):
            city_canonical.setdefault(_strip_accents(alias).upper(), canonical)
        for area, area_meta in meta.get('areas', {}).items():
            if area_meta.get('infer_city_allowed', False):
                ak = _strip_accents(area).upper()
                city_canonical.setdefault(ak, canonical)
                for aa in area_meta.get('aliases', []):
                    city_canonical.setdefault(_strip_accents(aa).upper(), canonical)

    # Hard-coded common abbreviations / known variants
    _ABBREVS = {
        'CSFP': 'CITY OF SAN FERNANDO', 'CDO': 'CAGAYAN DE ORO',
        'CDOC': 'CAGAYAN DE ORO', 'ZC': 'ZAMBOANGA CITY',
        'QC': 'QUEZON CITY', 'MNL': 'CITY OF MANILA',
        'BGC': 'CITY OF TAGUIG', 'GSC': 'GENERAL SANTOS',
        'CSJDM': 'CITY OF SAN JOSE DEL MONTE', 'PQUE': 'CITY OF PARANAQUE',
        'VAL CITY': 'CITY OF VALENZUELA', 'VALENZUELA': 'CITY OF VALENZUELA',
        'BALIUAG': 'CITY OF BALIWAG', 'BALIWAG': 'CITY OF BALIWAG',
        'DASMATINAS': 'CITY OF DASMARINAS', 'DASMARINAS': 'CITY OF DASMARINAS',
        'DASMARIÑAS': 'CITY OF DASMARINAS', 'PARANAQUE': 'CITY OF PARANAQUE',
        'LAS PINAS': 'CITY OF LAS PINAS', 'MONTALBAN': 'CITY OF RODRIGUEZ',
        'RODRIGUEZ': 'CITY OF RODRIGUEZ', 'MAYNILA': 'CITY OF MANILA',
        'MANILA CITY': 'CITY OF MANILA', 'KENNON ROAD': 'BAGUIO CITY',
        'CAMP 8': 'BAGUIO CITY',
    }
    for abbr, target in _ABBREVS.items():
        if target:
            resolved = city_canonical.get(target.upper(), target.upper())
            city_canonical.setdefault(abbr, resolved)

    # ── prov_canonical ────────────────────────────────────────────────────────
    prov_canonical: dict[str, str] = {p: p for p in unique_provs}
    for prov_key, aliases in area_dict.get('province_aliases', {}).items():
        pk = _strip_accents(prov_key).upper()
        prov_canonical.setdefault(pk, pk)
        for alias in aliases:
            prov_canonical.setdefault(_strip_accents(alias).upper(), pk)

    # ── relationship indexes ───────────────────────────────────────────────────
    city_to_prov: dict[str, list[str]] = {
        cn: grp['_prov_norm'].unique().tolist()
        for cn, grp in dim.groupby('_city_norm')
    }
    ambiguous_cities: set[str] = {c for c, provs in city_to_prov.items() if len(provs) > 1}

    prov_to_cities: dict[str, list[str]] = {
        pn: grp['_city_norm'].unique().tolist()
        for pn, grp in dim.groupby('_prov_norm')
    }

    brgy_to_locations: dict[str, list[tuple]] = {}
    for _, row in dim.iterrows():
        brgy_to_locations.setdefault(row['_brgy_norm'], []).append(
            (row['_city_norm'], row['_prov_norm'], str(row['psgc_10_digit']))
        )

    city_brgy_index: dict[str, list[tuple[str, str]]] = {}
    for _, row in dim.iterrows():
        city_brgy_index.setdefault(row['_city_norm'], []).append(
            (row['_brgy_norm'], str(row['psgc_10_digit']))
        )

    log.info(
        f'{len(city_canonical)} city aliases | {len(prov_canonical)} prov aliases | '
        f'{len(ambiguous_cities)} ambiguous cities | {len(unique_brgys)} unique barangays'
    )

    return dict(
        dim=dim, alias_df=alias_df, area_dict=area_dict,
        unique_cities=unique_cities, unique_provs=unique_provs, unique_brgys=unique_brgys,
        city_canonical=city_canonical, prov_canonical=prov_canonical,
        city_to_prov=city_to_prov, ambiguous_cities=ambiguous_cities,
        prov_to_cities=prov_to_cities, brgy_to_locations=brgy_to_locations,
        city_brgy_index=city_brgy_index,
    )


REF = load_reference(DIM_LOCATION, ALIAS_FILE, AREA_DICT_FILE)

16:51:21  INFO      dim_location: 42,011 rows
16:51:21  INFO      alias: 514 pairs
16:51:21  INFO      area_dict v2.1.0 loaded
16:51:25  INFO      1751 city aliases | 112 prov aliases | 115 ambiguous cities | 25836 unique barangays


## Cell 4 — Stage 1: Normalization

Handles:
- PH noise tokens (Block, Lot, Unit, Phase, Blk, etc.)
- Concatenated/compressed text (`873RizalStAlbay` → `873 Rizal St Albay`)
- Alias expansion via alias table
- Street-token stripping for city matching

In [59]:
# ── Build alias regex from alias table ───────────────────────────────────────
def _sanitize_repl(s: str) -> str:
    s = str(s)
    _ALIAS_REPL_FIXES = {
        'DASMARI\ufffdAS CITY': 'DASMARINAS CITY',
        'DASMARI\ufffdAS': 'DASMARINAS',
        'PARA\ufffdAQUE CITY': 'PARANAQUE CITY',
        'PARA\ufffdAQUE': 'PARANAQUE',
        'LAS PI\ufffdAS CITY': 'LAS PINAS CITY',
        'LAS PI\ufffdAS': 'LAS PINAS',
    }
    for bad, good in _ALIAS_REPL_FIXES.items():
        s = s.replace(bad, good)
    s = s.replace('\ufffd', '')
    return _strip_accents(s).upper().strip()


def _build_alias_regex(alias_df: pd.DataFrame):
    pairs = [
        (str(p).strip().upper(), _sanitize_repl(r))
        for p, r in zip(alias_df['pattern'], alias_df['replacement'])
        if str(p).strip() and str(r).strip()
    ]
    pairs.sort(key=lambda x: -len(x[0]))   # longest first
    rmap = {p: r for p, r in pairs}
    pat  = '|'.join(re.escape(p) for p, _ in pairs)
    return re.compile(r'\b(' + pat + r')\b'), rmap


_ALIAS_RE, _ALIAS_MAP = _build_alias_regex(REF['alias_df'])

# ── PH noise tokens to strip ─────────────────────────────────────────────────
_NOISE_RE = re.compile(
    r'\b(BLOCK|BLK|BK|LOT|LT|UNIT|PHASE|PH|PUROK|PRK|'
    r'SITIO|STI|ZONE|ZN|SUBDIVISION|SUBD|VILLAGE|VLG|'
    r'TOWNHOUSE|TOWNHOMES|COMPOUND|CPND|COMPOUND|'
    r'NEW CITY|NEW TOWN|EXTENSION|EXT|PACKAGE|PKG|'
    r'NO|NO\.|#|NUMBER)\b',
    re.IGNORECASE,
)

# ── Street suffixes that indicate preceding token is a street name ───────────
_STREET_SUFFIXES = frozenset({
    'STREET', 'ST', 'AVENUE', 'AVE', 'BOULEVARD', 'BLVD',
    'ROAD', 'RD', 'DRIVE', 'DR', 'HIGHWAY', 'HWY', 'LANE', 'LN',
    'CIRCLE', 'CIR', 'ALLEY', 'CORNER', 'COR', 'EXTENSION', 'EXT',
})

# ── Concatenated word splitter (camelCase / title-run-together text) ─────────
_CAMEL_RE = re.compile(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])')

# ── Leading numbers (house/lot numbers) ──────────────────────────────────────
_LEADING_NUM_RE = re.compile(r'^\d+[A-Za-z]?\s+')


def split_concatenated(text: str) -> str:
    """
    Split concatenated text like '873RizalStAlbay' into '873 Rizal St Albay'.
    Also handles digit–letter and letter–digit boundaries.
    """
    # camelCase / TitleCase boundaries
    text = _CAMEL_RE.sub(' ', text)
    # digit→letter boundary: '873Rizal' → '873 Rizal'
    text = re.sub(r'(\d)([A-Za-z])', r'\1 \2', text)
    # letter→digit boundary: 'Blk14' → 'Blk 14'
    text = re.sub(r'([A-Za-z])(\d)', r'\1 \2', text)
    return text


def normalize(text: str) -> str:
    """
    Full normalization pipeline.
    Returns uppercase, accent-free, noise-reduced address string.
    """
    if not text or str(text).strip().lower() in ('nan', 'none', ''):
        return ''

    # 1. Split concatenated text before uppercasing
    text = split_concatenated(str(text))

    # 2. Strip accents + uppercase
    text = _strip_accents(text).upper().strip()

    # 3. Drop 'NAN' tokens
    text = re.sub(r'\bNAN\b', ' ', text)

    # 4. Replace non-alphanumeric (except spaces) with space
    text = re.sub(r'[^A-Z0-9\s]', ' ', text)

    # 5. Expand aliases (BRGY → BARANGAY, POB → POBLACION, etc.)
    text = _ALIAS_RE.sub(lambda m: _ALIAS_MAP[m.group(0)], text)

    # 6. Remove PH structural noise tokens
    text = _NOISE_RE.sub(' ', text)

    # 7. Deduplicate repeated tokens (e.g., 'ALBAY ALBAY')
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text)

    # 8. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def strip_street_tokens(norm_addr: str) -> str:
    """
    Remove street name + suffix pairs to avoid city name false-positives.
    'RIZAL STREET POBLACION VALLADOLID' → 'POBLACION VALLADOLID'
    """
    tokens = norm_addr.split()
    result = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i + 1] in _STREET_SUFFIXES:
            i += 2   # skip street-name token + suffix
        elif tokens[i] in _STREET_SUFFIXES:
            i += 1   # orphaned suffix
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)


# ── Smoke test ────────────────────────────────────────────────────────────────
print('Normalization tests:')
for raw in [
    'Block 13 Lot 25 Buenavista Townhomes',
    'BORACAY, ISLAND MALAY AKLAN',
    '873RizalStAlbayDistrict Bgy. 17 - Rizal Street., Ilawod (Pob.) LEGAZPI CITY ALBAY',
    'Blk 14 Lot 34 Imperial St. Manchester 11 extension, Lancaster New City',
    'tagalag val city',
]:
    n = normalize(raw)
    s = strip_street_tokens(n)
    print(f'  IN : {raw}')
    print(f'  NRM: {n}')
    print(f'  SST: {s}')
    print()

Normalization tests:
  IN : Block 13 Lot 25 Buenavista Townhomes
  NRM: 13 25 BUENAVISTA
  SST: 13 25 BUENAVISTA

  IN : BORACAY, ISLAND MALAY AKLAN
  NRM: BORACAY ISLAND MALAY AKLAN
  SST: BORACAY ISLAND MALAY AKLAN

  IN : 873RizalStAlbayDistrict Bgy. 17 - Rizal Street., Ilawod (Pob.) LEGAZPI CITY ALBAY
  NRM: 873 RIZAL STREET ALBAY DISTRICT BARANGAY 17 RIZAL STREET ILAWOD POBLACION LEGAZPI CITY ALBAY
  SST: 873 ALBAY DISTRICT BARANGAY 17 ILAWOD POBLACION LEGAZPI CITY ALBAY

  IN : Blk 14 Lot 34 Imperial St. Manchester 11 extension, Lancaster New City
  NRM: 14 34 IMPERIAL STREET MANCHESTER 11 LANCASTER
  SST: 14 34 MANCHESTER 11 LANCASTER

  IN : tagalag val city
  NRM: TAGALAG VAL CITY
  SST: TAGALAG VAL CITY



## Cell 5 — Stage 2: Query Generation

Progressive degradation strategy:
1. Full cleaned address + Philippines  
2. After removing house/unit numbers  
3. Strip noise tokens, keep geo tokens  
4. Best city + province only  
5. Province or landmark only + Philippines

In [60]:
# Tokens that are geography-bearing (worth keeping in queries)
_QUERY_STOPWORDS = frozenset({
    'CITY', 'MUNICIPALITY', 'BARANGAY', 'PROVINCE', 'REGION',
    'STREET', 'AVE', 'BLVD', 'RD', 'DR', 'HWY',
    'CORNER', 'COR', 'AND', 'THE', 'AREA', 'DISTRICT',
})

_GEO_ANCHOR_TOKENS = frozenset({
    'BARANGAY', 'CITY', 'MUNICIPALITY', 'PROVINCE',
    'ISLAND', 'DISTRICT', 'POBLACION', 'VILLAGE',
})

# Heavy structural noise (not geography)
_STRUCTURAL_NOISE = re.compile(
    r'\b(BLOCK|BLK|LOT|UNIT|PHASE|PUROK|SITIO|ZONE|'
    r'SUBDIVISION|SUBD|COMPOUND|PACKAGE|EXTENSION|EXT|'
    r'TOWNHOMES|TOWNHOUSE|APARTMENT|APT|FLOOR|FL|'
    r'BUILDING|BLDG|ROOM|RM|SUITE|STE)\b',
    re.IGNORECASE,
)

_NUMBER_PREFIX_RE = re.compile(r'^\s*\d+[A-Z]?\s+')


def _clean_query(text: str) -> str:
    """Strip leading numbers and collapse spaces for a clean query string."""
    text = re.sub(r'^(\d+\s+)+', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def _extract_geo_tokens(norm_addr: str) -> str:
    """
    Keep only meaningful geographic tokens (4+ chars, not structural noise).
    This forms the 'minimal geo' query variant.
    """
    tokens = norm_addr.split()
    geo = []
    for tok in tokens:
        # Keep anchor tokens as-is
        if tok in _GEO_ANCHOR_TOKENS:
            geo.append(tok)
            continue
        # Keep if 4+ chars and not a noise token
        if len(tok) >= 4 and not _STRUCTURAL_NOISE.match(tok) and not tok.isdigit():
            geo.append(tok)
    return ' '.join(geo)


def generate_queries(raw: str, norm: str) -> list[str]:
    """
    Generate up to MAX_QUERY_VARIANTS progressive fallback queries.

    Strategy (most specific → most general):
    Q1: Full raw cleaned + Philippines        (catches well-formatted addresses)
    Q2: Normalized, no leading numbers        (removes house numbers)
    Q3: Geo tokens only + Philippines         (strips structural noise)
    Q4: Last N meaningful tokens + Philippines (drops subdivision names)
    Q5: Single place name + Philippines        (ultimate fallback)
    """
    ph = NOMINATIM_COUNTRY

    # FIX 5: resolve city abbreviations to full names before building Nominatim queries.
    # This ensures 'VAL CITY' → 'CITY OF VALENZUELA', 'DASMA' → 'DASMARINAS', etc.
    def _resolve_abbrevs(text: str) -> str:
        toks = text.split()
        for size in range(min(3, len(toks)), 0, -1):
            for i in range(len(toks) - size + 1):
                phrase = ' '.join(toks[i:i + size])
                canon = REF['city_canonical'].get(phrase)
                if canon and canon != phrase:  # real abbreviation
                    toks = toks[:i] + [canon] + toks[i + size:]
                    return ' '.join(toks)  # one substitution per pass
        return text

    norm_resolved = _resolve_abbrevs(norm)

    queries: list[str] = []
    seen: set[str] = set()

    def _add(q: str) -> None:
        q = q.strip()
        if q and q not in seen:
            seen.add(q)
            queries.append(q)

    # Q1: Raw cleaned (preserve original casing for proper names)
    raw_clean = re.sub(r'[^\w\s,.]', ' ', str(raw))
    raw_clean = re.sub(r'\s+', ' ', raw_clean).strip()
    _add(f'{raw_clean}, {ph}')

    # Q2: FIX 2 — strip ALL leading bare numeric tokens.
    # NOISE_RE removes 'Block'/'Lot' keywords but leaves bare numbers '13 25'.
    norm_clean = re.sub(r'^(\d+\s+)+', '', norm).strip()
    norm_clean = _clean_query(norm_clean)
    _add(f'{norm_clean}, {ph}')

    # Q3: Geo tokens only — from abbreviation-resolved norm
    geo = _extract_geo_tokens(norm_resolved)
    if geo:
        _add(f'{geo}, {ph}')

    # Q4/Q5: FIX 3 — exclude stopwords (CITY, BARANGAY, etc.) from tail selection.
    # FIX 5 — use norm_resolved so abbreviations become full city names.
    tokens = [t for t in norm_resolved.split()
              if len(t) >= 3 and not t.isdigit() and t not in _QUERY_STOPWORDS]
    if len(tokens) > 4:
        tail = ' '.join(tokens[-4:])
        _add(f'{tail}, {ph}')
    elif tokens:
        _add(f'{" ".join(tokens)}, {ph}')

    # Q5: Last 2 tokens (bare location)
    if len(tokens) >= 2:
        last2 = ' '.join(tokens[-2:])
        _add(f'{last2}, {ph}')
    if tokens:
        _add(f'{tokens[-1]}, {ph}')

    return queries[:MAX_QUERY_VARIANTS]


# ── Test query generation ─────────────────────────────────────────────────────
print('Query generation tests:')
for raw in [
    'Block 13 Lot 25 Buenavista Townhomes',
    'BORACAY, ISLAND MALAY AKLAN',
    '873RizalStAlbayDistrict Bgy. 17 - Rizal Street., Ilawod (Pob.) LEGAZPI CITY ALBAY',
]:
    norm = normalize(raw)
    qs = generate_queries(raw, norm)
    print(f'IN: {raw}')
    for i, q in enumerate(qs, 1):
        print(f'  Q{i}: {q}')
    print()

Query generation tests:
IN: Block 13 Lot 25 Buenavista Townhomes
  Q1: Block 13 Lot 25 Buenavista Townhomes, Philippines
  Q2: BUENAVISTA, Philippines

IN: BORACAY, ISLAND MALAY AKLAN
  Q1: BORACAY, ISLAND MALAY AKLAN, Philippines
  Q2: BORACAY ISLAND MALAY AKLAN, Philippines
  Q3: MALAY AKLAN, Philippines
  Q4: AKLAN, Philippines

IN: 873RizalStAlbayDistrict Bgy. 17 - Rizal Street., Ilawod (Pob.) LEGAZPI CITY ALBAY
  Q1: 873RizalStAlbayDistrict Bgy. 17 Rizal Street., Ilawod Pob. LEGAZPI CITY ALBAY, Philippines
  Q2: RIZAL STREET ALBAY DISTRICT BARANGAY 17 RIZAL STREET ILAWOD POBLACION LEGAZPI CITY ALBAY, Philippines
  Q3: RIZAL STREET ALBAY DISTRICT BARANGAY RIZAL STREET ILAWOD POBLACION CITY LEGAZPI CITY ALBAY, Philippines
  Q4: ILAWOD POBLACION LEGAZPI ALBAY, Philippines
  Q5: LEGAZPI ALBAY, Philippines



## Cell 6 — Stage 3: Nominatim Geocoding Engine

In [61]:
# ── FIX 1: SSL — suppress cert verification errors ─────────────────────────
# Some environments (corporate proxies, Docker, cloud VMs) lack a CA bundle
# trusted for nominatim.openstreetmap.org. Set NOMINATIM_VERIFY_SSL=True if
# your environment has valid certificates.
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
NOMINATIM_VERIFY_SSL = False

# ── Session with proper headers ──────────────────────────────────────────────
_SESSION = requests.Session()
_SESSION.headers.update({
    'User-Agent': NOMINATIM_USER_AGENT,
    'Accept-Language': 'en',
})

# ── In-memory cache (query string → result dict | None) ──────────────────────
_NOMINATIM_CACHE: dict[str, Optional[dict]] = {}

# ── Rate limiting: track last request time ───────────────────────────────────
_last_request_time = 0.0


def _cache_get(key: str) -> tuple[bool, Optional[dict]]:
    """Check in-memory then disk cache. Returns (hit, value)."""
    if key in _NOMINATIM_CACHE:
        return True, _NOMINATIM_CACHE[key]
    if _disk_cache is not None:
        try:
            __sentinel = object()
            val = _disk_cache.get(key, default=__sentinel)
            if val is not __sentinel:
                _NOMINATIM_CACHE[key] = val   # warm in-memory cache
                return True, val
        except Exception:
            pass
    return False, None


def _cache_set(key: str, value: Optional[dict]) -> None:
    _NOMINATIM_CACHE[key] = value
    if _disk_cache is not None:
        try:
            _disk_cache.set(key, value)
        except Exception:
            pass


def _nominatim_single(query: str) -> Optional[dict]:
    """
    Call Nominatim for a single query string.
    Returns top result dict or None.
    Respects rate limit and caching.
    """
    global _last_request_time

    cache_key = hashlib.md5(query.lower().encode()).hexdigest()
    hit, cached = _cache_get(cache_key)
    if hit:
        return cached

    # Rate limit enforcement
    elapsed = time.time() - _last_request_time
    if elapsed < NOMINATIM_RATE_LIMIT:
        time.sleep(NOMINATIM_RATE_LIMIT - elapsed)

    params = {
        'q': query,
        'format': 'json',
        'limit': 1,
        'countrycodes': 'ph',   # restrict to Philippines at API level
        'addressdetails': 1,
    }

    try:
        _last_request_time = time.time()
        r = _SESSION.get(
            NOMINATIM_URL,
            params=params,
            timeout=NOMINATIM_TIMEOUT,
            verify=NOMINATIM_VERIFY_SSL,
        )
        r.raise_for_status()
        results = r.json()
        result = results[0] if results else None
    except requests.RequestException as e:
        log.warning(f'Nominatim error for {query!r}: {e}')
        result = None

    _cache_set(cache_key, result)
    return result


def geocode_nominatim(queries: list[str]) -> Optional[dict]:
    """
    Try each query in order; stop and return on first success.
    Returns a normalized result dict or None.
    """
    for query in queries:
        raw_result = _nominatim_single(query)
        if raw_result:
            return {
                'source':       'nominatim',
                'query_used':   query,
                'lat':          raw_result.get('lat'),
                'lon':          raw_result.get('lon'),
                'display_name': raw_result.get('display_name'),
                'osm_type':     raw_result.get('osm_type'),
                'osm_id':       raw_result.get('osm_id'),
                'place_rank':   raw_result.get('place_rank'),
                'importance':   raw_result.get('importance'),
            }
    return None


print('Nominatim engine defined.')
print(f'Rate limit: {NOMINATIM_RATE_LIMIT}s | Timeout: {NOMINATIM_TIMEOUT}s')
print(f'Cache: in-memory + {"disk" if _disk_cache else "none"}')

Nominatim engine defined.
Rate limit: 1.05s | Timeout: 10s
Cache: in-memory + none


## Cell 7 — Stage 4: Fuzzy Local Dictionary Fallback

When all Nominatim queries fail, match against the local place dictionary
(barangays → cities → provinces) using RapidFuzz. Returns approximate
coordinates derived from PSGC centroid data (if available).

In [62]:
# ── Province/city approximate centroids ──────────────────────────────────────
# These are approximate geographic centers for major PH areas.
# In production, replace with a proper centroid table joined from PAGASA / PSA.
# Format: canonical_dim_key → (lat, lon)
_PLACE_CENTROIDS: dict[str, tuple[float, float]] = {
    # NCR cities
    'CITY OF MANILA':          (14.5995, 120.9842),
    'CITY OF QUEZON':          (14.6760, 121.0437),
    'CITY OF MAKATI':          (14.5547, 121.0244),
    'CITY OF TAGUIG':          (14.5176, 121.0509),
    'CITY OF PASIG':           (14.5764, 121.0851),
    'CITY OF MANDALUYONG':     (14.5794, 121.0359),
    'CITY OF MARIKINA':        (14.6507, 121.1029),
    'CITY OF CALOOCAN':        (14.7499, 120.9797),
    'CITY OF PARANAQUE':       (14.4793, 121.0198),
    'CITY OF LAS PINAS':       (14.4454, 120.9832),
    'CITY OF MUNTINLUPA':      (14.4081, 121.0415),
    'CITY OF VALENZUELA':      (14.7011, 120.9830),
    'CITY OF MALABON':         (14.6693, 120.9566),
    'CITY OF NAVOTAS':         (14.6669, 120.9427),
    'CITY OF PATEROS':         (14.5447, 121.0668),
    # Luzon
    'CITY OF SAN FERNANDO':    (15.0286, 120.6900),
    'CITY OF ANGELES':         (15.1450, 120.5887),
    'BAGUIO CITY':             (16.4023, 120.5960),
    'CITY OF DAGUPAN':         (16.0430, 120.3340),
    'CITY OF LEGAZPI':         (13.1391, 123.7438),
    'CITY OF NAGA':            (13.6218, 123.1948),
    'CITY OF LUCENA':          (13.9373, 121.6170),
    'CITY OF BATANGAS':        (13.7565, 121.0584),
    'CITY OF DASMARINAS':      (14.2930, 120.9390),
    'CITY OF BACOOR':          (14.4580, 120.9340),
    'CITY OF IMUS':            (14.4297, 120.9367),
    'CITY OF CAVITE':          (14.4791, 120.8978),
    'CITY OF SAN JOSE DEL MONTE': (14.8137, 121.0449),
    'CITY OF MEYCAUAYAN':      (14.7363, 120.9614),
    'CITY OF MALOLOS':         (14.8527, 120.8127),
    'CITY OF BALIWAG':         (14.9522, 120.9047),
    'CITY OF CABANATUAN':      (15.4875, 120.9736),
    'CITY OF RODRIGUEZ':       (14.7357, 121.1153),
    # Visayas
    'CITY OF CEBU':            (10.3157, 123.8854),
    'CITY OF MANDAUE':         (10.3236, 123.9223),
    'CITY OF LAPU-LAPU':       (10.3119, 123.9494),
    'CITY OF ILOILO':          (10.7202, 122.5621),
    'CITY OF BACOLOD':         (10.6407, 122.9658),
    'CITY OF TACLOBAN':        (11.2543, 125.0000),
    'MALAY':                   (11.9048, 121.9199),   # Boracay
    # Mindanao
    'CITY OF DAVAO':           (7.1907, 125.4553),
    'CITY OF GENERAL SANTOS':  (6.1164, 125.1716),
    'CAGAYAN DE ORO':          (8.4542, 124.6319),
    'ZAMBOANGA CITY':          (6.9214, 122.0790),
    'CITY OF ILIGAN':          (8.2280, 124.2452),
    'CITY OF COTABATO':        (7.2047, 124.2310),
    'CITY OF BUTUAN':          (8.9475, 125.5406),
    # Provinces (approximate center)
    'METRO MANILA':            (14.5995, 121.0000),
    'CEBU':                    (10.3157, 123.8854),
    'DAVAO DEL SUR':           (6.8000, 125.4000),
    'ALBAY':                   (13.1780, 123.5280),
    'CAVITE':                  (14.2809, 120.8633),
    'BULACAN':                 (14.7942, 120.8760),
    'LAGUNA':                  (14.1407, 121.4673),
    'RIZAL':                   (14.5995, 121.1778),
    'PAMPANGA':                (15.0794, 120.6200),
    'BATANGAS':                (13.7565, 121.0584),
    'AKLAN':                   (11.8166, 122.0942),
    'LEYTE':                   (10.8624, 124.8811),
}


def _fuzzy_match_place(
    norm_addr: str,
    candidates: list[str],
    threshold: float = 70.0,
) -> tuple[Optional[str], float]:
    """Return (best_match, score) or (None, 0) if below threshold."""
    if not candidates:
        return None, 0.0
    result = rf_process.extractOne(
        norm_addr, candidates,
        scorer=fuzz.partial_ratio,
        score_cutoff=threshold,
    )
    if result:
        return result[0], float(result[1])
    return None, 0.0


def geocode_fuzzy_fallback(raw: str, norm: str) -> Optional[dict]:
    """
    Fuzzy match against the local place dictionary.
    Hierarchy: barangay (within city) → city → province.
    Returns a result dict or None.
    """
    city_canonical = REF['city_canonical']
    prov_canonical = REF['prov_canonical']
    unique_cities  = REF['unique_cities']
    unique_provs   = REF['unique_provs']
    city_brgy_index = REF['city_brgy_index']
    city_to_prov   = REF['city_to_prov']

    street_stripped = strip_street_tokens(norm)

    # ── Step 1: match city ───────────────────────────────────────────────────
    matched_city: Optional[str] = None
    city_score: float = 0.0

    # Exact n-gram scan
    tokens = street_stripped.split()
    for size in range(min(6, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i:i + size])
            if phrase in city_canonical:
                matched_city = city_canonical[phrase]
                city_score = 100.0
                break
        if matched_city:
            break

    # Per-token fuzzy scan (catches subdivision names like 'LANCASTER')
    if not matched_city:
        best_city, best_score = None, 0.0
        scan_toks = [t for t in tokens if len(t) >= 4 and not t.isdigit()]
        for tok in scan_toks:
            mc, ms = _fuzzy_match_place(tok, unique_cities, threshold=FUZZY_CITY_HIGH)
            if mc and ms > best_score:
                best_city, best_score = mc, ms
        if best_city:
            matched_city, city_score = best_city, best_score
        else:
            # Last resort: whole-string fuzzy
            mc, city_score = _fuzzy_match_place(street_stripped, unique_cities, threshold=FUZZY_CITY_LOW)
            if mc:
                matched_city = mc

    # ── Step 2: match province ───────────────────────────────────────────────
    matched_prov: Optional[str] = None
    prov_score: float = 0.0

    for size in range(min(4, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i:i + size])
            if phrase in prov_canonical:
                matched_prov = prov_canonical[phrase]
                prov_score = 100.0
                break
        if matched_prov:
            break

    if not matched_prov:
        mp, prov_score = _fuzzy_match_place(norm, unique_provs, threshold=FUZZY_PROV_LOW)
        if mp:
            matched_prov = mp

    # ── Step 3: match barangay within city (strict to reduce false positives) ─
    matched_brgy: Optional[str] = None
    brgy_score: float = 0.0

    if matched_city:
        entries = city_brgy_index.get(matched_city, [])
        if entries:
            brgy_names = [e[0] for e in entries]
            brgy_set = set(brgy_names)

            # 3a) exact n-gram search first
            for size in range(min(4, len(tokens)), 0, -1):
                for i in range(len(tokens) - size + 1):
                    phrase = ' '.join(tokens[i:i + size])
                    if phrase in brgy_set:
                        matched_brgy = phrase
                        brgy_score = 100.0
                        break
                if matched_brgy:
                    break

            # 3b) token-level fuzzy with very high threshold
            if not matched_brgy:
                best_b, best_s = None, 0.0
                for tok in [t for t in tokens if len(t) >= 4 and not t.isdigit()]:
                    mb, ms = _fuzzy_match_place(tok, brgy_names, threshold=max(FUZZY_BRGY_HIGH, 92))
                    if mb and ms > best_s:
                        best_b, best_s = mb, ms
                if best_b:
                    matched_brgy, brgy_score = best_b, best_s

            # 3c) full-string fallback with stricter threshold than before
            if not matched_brgy:
                mb, brgy_score = _fuzzy_match_place(
                    street_stripped,
                    brgy_names,
                    threshold=max(FUZZY_BRGY_HIGH + 4, 92),
                )
                if mb:
                    matched_brgy = mb

    # ── Step 4: resolve coordinates ──────────────────────────────────────────
    if not matched_city and not matched_prov:
        return None

    # Look up coordinates from centroid table
    coord_key = matched_city or matched_prov or ''
    coords = _PLACE_CENTROIDS.get(coord_key)
    if not coords and matched_prov:
        coords = _PLACE_CENTROIDS.get(matched_prov)

    lat = str(coords[0]) if coords else None
    lon = str(coords[1]) if coords else None

    # Resolve display name from dim table
    dim = REF['dim']
    subset = dim[dim['_city_norm'] == matched_city] if matched_city else pd.DataFrame()
    if matched_prov and not subset.empty:
        subset = subset[subset['_prov_norm'] == matched_prov]

    city_display = subset['city_name'].iloc[0] if not subset.empty else matched_city
    prov_display = subset['province_name'].iloc[0] if not subset.empty else matched_prov
    reg_display  = subset['region_name'].iloc[0] if not subset.empty else None

    return {
        'source':         'fuzzy_dictionary',
        'matched_place':  city_display or prov_display,
        'matched_city':   city_display,
        'matched_prov':   prov_display,
        'matched_region': reg_display,
        'matched_brgy':   matched_brgy,
        'city_score':     round(city_score, 1),
        'prov_score':     round(prov_score, 1),
        'brgy_score':     round(brgy_score, 1),
        'lat':            lat,
        'lon':            lon,
        'coords_approx':  True,
    }


print('Fuzzy fallback engine defined.')
print(f'Centroid coverage: {len(_PLACE_CENTROIDS)} known places')

Fuzzy fallback engine defined.
Centroid coverage: 59 known places


In [63]:
# ── Address quality + explicit-geo constrained fuzzy matching override ─────────
MIN_ADDRESS_CHARS = 15

_STRUCTURAL_ONLY_TOKENS = {
    'BLOCK', 'BLK', 'LOT', 'UNIT', 'PHASE', 'BLDG', 'BUILDING', 'TOWER',
    'TOWNHOUSE', 'TOWNHOMES', 'SUBDIVISION', 'SUBD', 'VILLAGE', 'COMPOUND',
    'PKG', 'PACKAGE', 'EXTENSION', 'EXT', 'ROOM', 'RM', 'FLOOR', 'FL',
}

_RAW_STREET_SUFFIXES = {
    'ST', 'STREET', 'AVE', 'AVENUE', 'BLVD', 'BOULEVARD',
    'RD', 'ROAD', 'DR', 'DRIVE', 'HWY', 'HIGHWAY', 'LANE', 'LN',
}

_EXPLICIT_CITY_FALLBACK = {
    'TONDO': 'TONDO I/II',
    'PARANAQUE': 'CITY OF PARANAQUE',
    'PQUE': 'CITY OF PARANAQUE',
    'QUEZON CITY': 'CITY OF QUEZON',
    'QC': 'CITY OF QUEZON',
    'VALENZUELA': 'CITY OF VALENZUELA',
    'LAS PINAS': 'CITY OF LAS PINAS',
    'LEGAZPI': 'CITY OF LEGAZPI',
}

_EXPLICIT_PROV_FALLBACK = {
    'MANILA': 'METRO MANILA',
    'NCR': 'METRO MANILA',
    'CAVITE': 'CAVITE',
    'BULACAN': 'BULACAN',
    'ALBAY': 'ALBAY',
    'AKLAN': 'AKLAN',
}


def _alnum_len(text: str) -> int:
    cleaned = re.sub(r'[^A-Z0-9]', '', _strip_accents(str(text)).upper())
    return len(cleaned)


def _raw_tokens(text: str) -> list[str]:
    up = _strip_accents(str(text)).upper()
    up = re.sub(r'[^A-Z0-9\s]', ' ', up)
    return [t for t in up.split() if t]


def _resolve_city_to_reference(text: str) -> Optional[str]:
    key = _strip_accents(str(text)).upper().strip()
    valid_cities = set(REF['unique_cities'])
    city_canonical = REF['city_canonical']

    if key in valid_cities:
        return key
    if key in city_canonical and city_canonical[key] in valid_cities:
        return city_canonical[key]

    matches = [c for c in valid_cities if key in c or c in key]
    if matches:
        return sorted(matches, key=len)[0]
    return None


def _resolve_prov_to_reference(text: str) -> Optional[str]:
    key = _strip_accents(str(text)).upper().strip()
    valid_provs = set(REF['unique_provs'])
    prov_canonical = REF['prov_canonical']

    if key in valid_provs:
        return key
    if key in prov_canonical and prov_canonical[key] in valid_provs:
        return prov_canonical[key]

    matches = [p for p in valid_provs if key in p or p in key]
    if matches:
        return sorted(matches, key=len)[0]
    return None


def extract_explicit_geography(raw: str, norm: str) -> dict:
    """
    Conservative explicit-geo extraction:
    - trusts high-confidence fallback keywords first
    - falls back to full official names (2+ words) only when no city fallback hit
    """
    raw_up = ' '.join(_raw_tokens(raw))
    raw_toks = raw_up.split()

    blocked_single_tokens = set()
    for i in range(len(raw_toks) - 1):
        if raw_toks[i + 1] in _RAW_STREET_SUFFIXES:
            blocked_single_tokens.add(raw_toks[i])

    fallback_cities: list[str] = []
    explicit_cities: list[str] = []
    explicit_provs: list[str] = []

    for k, v in _EXPLICIT_CITY_FALLBACK.items():
        if k in raw_up:
            resolved = _resolve_city_to_reference(v)
            if resolved:
                fallback_cities.append(resolved)

    for k, v in _EXPLICIT_PROV_FALLBACK.items():
        if k in raw_up:
            resolved = _resolve_prov_to_reference(v)
            if resolved:
                explicit_provs.append(resolved)

    # If fallback cities exist, prioritize them over generic official-name scanning.
    if fallback_cities:
        explicit_cities.extend(fallback_cities)
    else:
        for city in REF['unique_cities']:
            if len(city.split()) >= 2 and city in raw_up:
                explicit_cities.append(city)

    for prov in REF['unique_provs']:
        if len(prov.split()) >= 2 and prov in raw_up:
            explicit_provs.append(prov)

    filtered_cities = []
    for c in dict.fromkeys(explicit_cities):
        if len(c.split()) == 1 and c in blocked_single_tokens:
            continue
        filtered_cities.append(c)

    explicit_cities = filtered_cities
    explicit_provs = list(dict.fromkeys(explicit_provs))

    return {
        'explicit_cities': explicit_cities,
        'explicit_provs': explicit_provs,
        'has_explicit_geography': bool(explicit_cities or explicit_provs),
    }


def classify_ambiguous_address(raw: str, norm: str) -> tuple[bool, Optional[str]]:
    """
    Return (is_ambiguous, reason) for addresses likely to produce false positives.
    Rules:
    - very short addresses
    - structure/street-only addresses with no explicit geo signal
    """
    if _alnum_len(raw) < MIN_ADDRESS_CHARS:
        return True, 'short_lt_15'

    explicit_geo = extract_explicit_geography(raw, norm)
    if explicit_geo['has_explicit_geography']:
        return False, None

    toks = _raw_tokens(raw)
    if not toks:
        return True, 'empty_after_normalize'

    street_like = any(t in _RAW_STREET_SUFFIXES for t in toks)
    structural_hits = [t for t in toks if t in _STRUCTURAL_ONLY_TOKENS]

    core = [
        t for t in toks
        if not t.isdigit()
        and t not in _STRUCTURAL_ONLY_TOKENS
        and t not in _RAW_STREET_SUFFIXES
        and len(t) >= 4
    ]

    if structural_hits and len(core) <= 3:
        return True, 'structure_only_no_explicit_geo'
    if street_like and len(core) <= 3:
        return True, 'street_only_no_explicit_geo'

    return False, None


def _street_name_tokens(tokens: list[str]) -> set[str]:
    """Tokens directly before a street suffix (e.g., IPIL ST) should not be place-matched."""
    blocked = set()
    for i in range(len(tokens) - 1):
        if tokens[i + 1] in _STREET_SUFFIXES:
            blocked.add(tokens[i])
    return blocked


def geocode_fuzzy_fallback(raw: str, norm: str) -> Optional[dict]:
    """
    Constrained fuzzy matcher:
    - respects explicit city/province mentions when present
    - avoids matching street-name tokens as places
    """
    city_canonical = REF['city_canonical']
    prov_canonical = REF['prov_canonical']
    unique_cities = REF['unique_cities']
    unique_provs = REF['unique_provs']
    city_brgy_index = REF['city_brgy_index']
    city_to_prov = REF['city_to_prov']

    street_stripped = strip_street_tokens(norm)
    tokens = street_stripped.split()
    blocked_place_tokens = _street_name_tokens(tokens)

    explicit_geo = extract_explicit_geography(raw, norm)
    explicit_cities = explicit_geo['explicit_cities']
    explicit_provs = explicit_geo['explicit_provs']

    city_candidates = unique_cities
    prov_candidates = unique_provs

    if explicit_cities:
        city_candidates = explicit_cities
        inferred_provs = []
        for c in explicit_cities:
            inferred_provs.extend(city_to_prov.get(c, []))
        if inferred_provs:
            prov_candidates = list(dict.fromkeys(inferred_provs))
    elif explicit_provs:
        prov_candidates = explicit_provs
        city_candidates = [
            c for c in unique_cities
            if any(p in explicit_provs for p in city_to_prov.get(c, []))
        ] or unique_cities

    matched_city: Optional[str] = None
    city_score: float = 0.0

    candidate_city_set = set(city_candidates)
    for size in range(min(6, len(tokens)), 0, -1):
        for i in range(len(tokens) - size + 1):
            phrase = ' '.join(tokens[i:i + size])
            if phrase in city_canonical:
                can_city = city_canonical[phrase]
                if can_city in candidate_city_set:
                    matched_city = can_city
                    city_score = 100.0
                    break
        if matched_city:
            break

    if not matched_city:
        best_city, best_score = None, 0.0
        scan_toks = [
            t for t in tokens
            if len(t) >= 4 and not t.isdigit() and t not in blocked_place_tokens
        ]
        for tok in scan_toks:
            mc, ms = _fuzzy_match_place(tok, city_candidates, threshold=FUZZY_CITY_HIGH)
            if mc and ms > best_score:
                best_city, best_score = mc, ms
        if best_city:
            matched_city, city_score = best_city, best_score
        else:
            mc, city_score = _fuzzy_match_place(street_stripped, city_candidates, threshold=FUZZY_CITY_LOW)
            if mc:
                matched_city = mc

    matched_prov: Optional[str] = None
    prov_score: float = 0.0

    if matched_city:
        provs = city_to_prov.get(matched_city, [])
        if len(provs) == 1:
            matched_prov = provs[0]
            prov_score = 100.0

    if not matched_prov:
        for size in range(min(4, len(tokens)), 0, -1):
            for i in range(len(tokens) - size + 1):
                phrase = ' '.join(tokens[i:i + size])
                if phrase in prov_canonical:
                    can_prov = prov_canonical[phrase]
                    if can_prov in prov_candidates:
                        matched_prov = can_prov
                        prov_score = 100.0
                        break
            if matched_prov:
                break

    if not matched_prov:
        mp, prov_score = _fuzzy_match_place(norm, prov_candidates, threshold=FUZZY_PROV_LOW)
        if mp:
            matched_prov = mp

    matched_brgy: Optional[str] = None
    brgy_score: float = 0.0

    if matched_city:
        entries = city_brgy_index.get(matched_city, [])
        if entries:
            brgy_names = [e[0] for e in entries]
            brgy_set = set(brgy_names)

            for size in range(min(4, len(tokens)), 0, -1):
                for i in range(len(tokens) - size + 1):
                    phrase = ' '.join(tokens[i:i + size])
                    if phrase in brgy_set:
                        matched_brgy = phrase
                        brgy_score = 100.0
                        break
                if matched_brgy:
                    break

            if not matched_brgy:
                best_b, best_s = None, 0.0
                for tok in [t for t in tokens if len(t) >= 4 and not t.isdigit() and t not in blocked_place_tokens]:
                    mb, ms = _fuzzy_match_place(tok, brgy_names, threshold=max(FUZZY_BRGY_HIGH, 92))
                    if mb and ms > best_s:
                        best_b, best_s = mb, ms
                if best_b:
                    matched_brgy, brgy_score = best_b, best_s

    if not matched_city and not matched_prov:
        return None

    coord_key = matched_city or matched_prov or ''
    coords = _PLACE_CENTROIDS.get(coord_key)
    if not coords and matched_prov:
        coords = _PLACE_CENTROIDS.get(matched_prov)

    lat = str(coords[0]) if coords else None
    lon = str(coords[1]) if coords else None

    dim = REF['dim']
    subset = dim[dim['_city_norm'] == matched_city] if matched_city else pd.DataFrame()
    if matched_prov and not subset.empty:
        subset = subset[subset['_prov_norm'] == matched_prov]

    city_display = subset['city_name'].iloc[0] if not subset.empty else matched_city
    prov_display = subset['province_name'].iloc[0] if not subset.empty else matched_prov
    reg_display = subset['region_name'].iloc[0] if not subset.empty else None

    return {
        'source': 'fuzzy_dictionary',
        'matched_place': city_display or prov_display,
        'matched_city': city_display,
        'matched_prov': prov_display,
        'matched_region': reg_display,
        'matched_brgy': matched_brgy,
        'city_score': round(city_score, 1),
        'prov_score': round(prov_score, 1),
        'brgy_score': round(brgy_score, 1),
        'lat': lat,
        'lon': lon,
        'coords_approx': True,
        'explicit_geography_found': explicit_geo['has_explicit_geography'],
    }


print('Address quality + constrained fuzzy override loaded.')

Address quality + constrained fuzzy override loaded.


In [64]:
# Keep a stable reference so later experimental cells don't override active pipeline behavior.
_extract_explicit_geography_v3 = extract_explicit_geography

## Cell 8 — Stage 5: Pipeline Orchestrator

Single-address entry point following the full flow:
`normalize → generate queries → Nominatim → fuzzy fallback → null`

In [65]:
def _null_result(raw: str, norm: str) -> dict:
    return {
        'input':          raw,
        'normalized':     norm,
        'source':         'none',
        'query_used':     None,
        'lat':            None,
        'lon':            None,
        'display_name':   None,
        'matched_place':  None,
        'matched_city':   None,
        'matched_prov':   None,
        'matched_region': None,
        'matched_brgy':   None,
        'city_score':     None,
        'prov_score':     None,
        'brgy_score':     None,
        'coords_approx':  None,
        'osm_type':       None,
        'osm_id':         None,
        'place_rank':     None,
        'importance':     None,
        'confidence':     'none',
        'result':         None,
        'drop_reason':    None,
    }


def process_address(raw: str) -> dict:
    """
    Full geocoding pipeline for a single address string.

    Returns one of three output shapes:
    - source='nominatim'         → Nominatim match
    - source='fuzzy_dictionary'  → local fuzzy match
    - source='none'              → no match / filtered
    """
    # ── Stage 1: Normalize ────────────────────────────────────────────────────
    norm = normalize(raw)
    out  = _null_result(raw, norm)

    if not norm:
        out['confidence'] = 'none'
        out['drop_reason'] = 'empty_after_normalize'
        return out

    # Gate low-quality ambiguous addresses before geocoding.
    is_ambiguous, reason = classify_ambiguous_address(raw, norm)
    if is_ambiguous:
        out['result'] = 'filtered'
        out['drop_reason'] = reason
        out['confidence'] = 'none'
        return out

    # ── Stage 2: Generate query variants ──────────────────────────────────────
    queries = generate_queries(raw, norm)

    # ── Stage 3: Try Nominatim ────────────────────────────────────────────────
    if ENABLE_NOMINATIM:
        nom_result = geocode_nominatim(queries)
        if nom_result:
            out.update(nom_result)
            # Confidence based on place_rank (lower = more specific)
            rank = nom_result.get('place_rank', 30)
            out['confidence'] = 'high' if rank <= 16 else 'medium' if rank <= 25 else 'low'
            out['result'] = 'ok'
            return out

    # ── Stage 4: Fuzzy local dictionary fallback ───────────────────────────────
    fuzz_result = geocode_fuzzy_fallback(raw, norm)
    if fuzz_result:
        out.update(fuzz_result)
        # Confidence based on match quality
        city_s = fuzz_result.get('city_score', 0) or 0
        out['confidence'] = (
            'high'   if city_s == 100 else
            'medium' if city_s >= FUZZY_CITY_HIGH else
            'low'
        )
        out['result'] = 'ok'
        return out

    # ── Stage 5: No match ────────────────────────────────────────────────────
    out['result'] = None
    return out


# ── Spot-check ────────────────────────────────────────────────────────────────
if not ENABLE_NOMINATIM:
    print('(Nominatim disabled — testing fuzzy fallback only)')

tests = [
    'Block 13 Lot 25 Buenavista Townhomes',
    'BORACAY, ISLAND MALAY AKLAN',
    '873RizalStAlbayDistrict Bgy. 17 - Rizal Street., Ilawod (Pob.) LEGAZPI CITY ALBAY',
    'Blk 14 Lot 34 Imperial St. Manchester 11 extension, Lancaster New City',
    'tagalag val city',
    'Salitran dasma cavite',
    'IPIL IPIL ST CAMPO SUBD',
    '4512 IPIL IPIL ST SAN ANTONIO PARANAQUE',
    '1330 TORRES BUGALLON ST COR IPIL TONDO MANILA',
]

print(f'{"INPUT":<55} {"SOURCE":<20} {"PLACE":<30} {"CONF":<8} {"DROP_REASON"}')
print('-' * 140)
for addr in tests:
    r = process_address(addr)
    place = r.get('matched_place') or (r.get('display_name') or '')[:40] or 'n/a'
    print(f'{addr[:53]:<55} {r["source"]:<20} {str(place)[:28]:<30} {r["confidence"]:<8} {str(r.get("drop_reason"))}')

(Nominatim disabled — testing fuzzy fallback only)
INPUT                                                   SOURCE               PLACE                          CONF     DROP_REASON
--------------------------------------------------------------------------------------------------------------------------------------------
Block 13 Lot 25 Buenavista Townhomes                    none                 n/a                            none     structure_only_no_explicit_geo
BORACAY, ISLAND MALAY AKLAN                             fuzzy_dictionary     Malay                          high     None
873RizalStAlbayDistrict Bgy. 17 - Rizal Street., Ilaw   fuzzy_dictionary     City of Legazpi                high     None
Blk 14 Lot 34 Imperial St. Manchester 11 extension, L   none                 n/a                            none     None
tagalag val city                                        none                 n/a                            none     short_lt_15
Salitran dasma cavite               

## BONUS: Interactive Address Debugger

Test any single address and see step-by-step how it's being matched. This helps identify problematic patterns.

In [66]:
def debug_address(test_address: str):
    """
    Interactive debugger: Shows exactly why an address gets matched.
    Displays: normalization, queries, fuzzy scores for each city/province candidate.
    """
    print('=' * 100)
    print(f'INPUT: {test_address}')
    print('=' * 100)
    
    # ── Stage 1: Normalization ────────────────────────────────────────────────
    norm = normalize(test_address)
    print(f'\n[STAGE 1] Normalization:')
    print(f'  Normalized: {norm}')
    
    street_stripped = strip_street_tokens(norm)
    print(f'  Street-tokens stripped: {street_stripped}')
    
    # Extract explicit city/province/barangay mentions from ORIGINAL input
    print(f'\n[STAGE 2] Explicit geo mentions in ORIGINAL address:')
    tokens = test_address.upper().split()
    city_keywords = ['CITY', 'MANILA', 'CEBU', 'DAVAO', 'QUEZON', 'BACOLOD', 'ILOILO', 'CDO', 'QC', 'BGC']
    prov_keywords = ['BULACAN', 'CAVITE', 'RIZAL', 'ALBAY', 'LEYTE', 'AKLAN', 'LAGUNA', 'PAMPANGA', 'BATANGAS', 'ISABELA', 'CEBU', 'DAVAO', 'ILOILO', 'CAPIZ']
    
    explicit_cities = [t for t in tokens if t in city_keywords]
    explicit_provs = [t for t in tokens if t in prov_keywords]
    
    if explicit_cities:
        print(f'  Explicit CITIES found: {explicit_cities}')
    if explicit_provs:
        print(f'  Explicit PROVINCES found: {explicit_provs}')
    if not explicit_cities and not explicit_provs:
        print(f'  [WARNING] NO explicit geographic markers in input!')
    
    # ── Stage 2: Query Generation ────────────────────────────────────────────
    queries = generate_queries(test_address, norm)
    print(f'\n[STAGE 3] Generated Queries (max {MAX_QUERY_VARIANTS}):')
    for i, q in enumerate(queries, 1):
        print(f'  Q{i}: {q}')
    
    # ── Stage 4: Fuzzy Matching Debug ────────────────────────────────────────
    print(f'\n[STAGE 4] Local Fuzzy Matching Analysis:')
    
    unique_cities = REF['unique_cities']
    unique_provs = REF['unique_provs']
    
    # Try matching city
    print(f'\n  CITY matching against {len(unique_cities)} candidates:')
    best_cities = rf_process.extract(street_stripped, unique_cities, scorer=fuzz.partial_ratio, limit=5, score_cutoff=70)
    if best_cities:
        for match_data in best_cities:
            city, score, idx = match_data
            print(f'    {city:<50} score={score:.1f}')
    else:
        print(f'    [NO MATCHES above score threshold]')
    
    # Try matching province
    print(f'\n  PROVINCE matching against {len(unique_provs)} candidates:')
    best_provs = rf_process.extract(norm, unique_provs, scorer=fuzz.partial_ratio, limit=5, score_cutoff=70)
    if best_provs:
        for match_data in best_provs:
            prov, score, idx = match_data
            print(f'    {prov:<50} score={score:.1f}')
    else:
        print(f'    [NO MATCHES above score threshold]')
    
    # ── Stage 5: Full Pipeline Result ────────────────────────────────────────
    print(f'\n[STAGE 5] Pipeline Result:')
    result = process_address(test_address)
    
    print(f'  Source: {result["source"]}')
    print(f'  Matched City: {result.get("matched_city", "N/A")}')
    print(f'  Matched Province: {result.get("matched_prov", "N/A")}')
    print(f'  Matched Barangay: {result.get("matched_brgy", "N/A")}')
    print(f'  City Score: {result.get("city_score", "N/A")}')
    print(f'  Prov Score: {result.get("prov_score", "N/A")}')
    print(f'  Brgy Score: {result.get("brgy_score", "N/A")}')
    print(f'  Confidence: {result.get("confidence", "N/A")}')
    print(f'  Coords: ({result.get("lat", "N/A")}, {result.get("lon", "N/A")})')
    
    if result.get("source") == "none":
        print(f'\n  [WARNING] NO MATCH FOUND')
    
    print('=' * 100)
    return result


# ── Test cases provided by user ──────────────────────────────────────────────
print('Interactive Debugger Ready!')
print('Call: debug_address("your address here")')
print()
print('Example problematic addresses to test:')
test_cases = [
    '1330 TORRES BUGALLON ST COR IPIL TONDO MANILA',  # Should be Manila, not Ipil
    'IPIL IPIL ST CAMPO SUBD',                          # Ambiguous - is this a street?
    '4512 IPIL IPIL ST WELCOME VILLAGE SAN ANTONIO VALLEY 1 PARANAQUE',  # Should be Paranaque
    'San Fernando Village',                              # Generic - which San Fernando?
    'Santo Domingo St',                                  # Street only, no city
    'BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VILLAGE,PUEBLO,BUSINESS PARK'  # Subdivision only
]
for addr in test_cases[:3]:  # Show first 3
    debug_address(addr)
    print('\n')

Interactive Debugger Ready!
Call: debug_address("your address here")

Example problematic addresses to test:
INPUT: 1330 TORRES BUGALLON ST COR IPIL TONDO MANILA

[STAGE 1] Normalization:
  Normalized: 1330 TORRES BUGALLON STREET CORNER IPIL CITY TONDO MANILA
  Street-tokens stripped: 1330 TORRES IPIL CITY TONDO MANILA

[STAGE 2] Explicit geo mentions in ORIGINAL address:
  Explicit CITIES found: ['MANILA']

[STAGE 3] Generated Queries (max 5):
  Q1: 1330 TORRES BUGALLON ST COR IPIL TONDO MANILA, Philippines
  Q2: TORRES BUGALLON STREET CORNER IPIL CITY TONDO MANILA, Philippines
  Q3: TORRES BUGALLON STREET CORNER IPIL CITY TONDO I/II MANILA, Philippines
  Q4: IPIL TONDO I/II MANILA, Philippines
  Q5: I/II MANILA, Philippines

[STAGE 4] Local Fuzzy Matching Analysis:

  CITY matching against 1426 candidates:
    IPIL                                               score=100.0
    ANILAO                                             score=90.9
    LILA                                       

## Proposed Fixes for Address Matching Issues

Based on the analysis above, here are the recommended improvements:

### Fix 1: Extract & Respect Explicit Geographic Markers
**Problem:** Addresses with explicit city/province mentions ("TONDO MANILA", "PARANAQUE") are being ignored.  
**Solution:** Extract city/province keywords from the raw input FIRST, then constrain fuzzy matching to only those geographies.

### Fix 2: Street Name vs Place Name Disambiguation  
**Problem:** "IPIL IPIL ST" is being matched to "Ipil, Zamboanga Sibugay" instead of recognizing IPIL as a street name.  
**Solution:** When a token appears with street suffixes (ST, STREET, AVE, etc.), suppress fuzzy matching against that token as a place name.

### Fix 3: Higher Confidence Requirement for Generic Names
**Problem:** "San Fernando Village", "Meralco Village", "Santo Domingo St" match ambiguously because these names exist in multiple places.  
**Solution:** For subdivision/village names without explicit city context, require VERY high match scores (>95) before accepting the match. Otherwise flag as uncertain.

### Fix 4: Flag High-Risk Incomplete Addresses
**Problem:** Short addresses (< 40 chars) with no city/province mention generate incorrect matches.  
**Solution:** Tag these with a "LOW_CONFIDENCE" flag in exports. Consider requiring manual review before using them.


In [67]:
# ─── FIX 1: EXTRACT EXPLICIT CITY/PROVINCE FROM RAW INPUT ───────────────────
# Before fuzzy matching, check if the address explicitly mentions a city/province.
# If so, constrain matching to ONLY that geography.

def extract_explicit_geography(raw_address: str) -> dict:
    """
    Extract explicit city/province keywords from the raw (original) address.
    Returns dict with 'explicit_cities' and 'explicit_provs' lists.
    """
    upper_addr = str(raw_address).upper()
    
    # Build a more comprehensive list of city/province keywords
    explicit_cities_map = {
        'MANILA': 'CITY OF MANILA',
        'QUEZON CITY': 'CITY OF QUEZON',
        'QUEZON': 'CITY OF QUEZON',
        'QC': 'CITY OF QUEZON',
        'MAKATI': 'CITY OF MAKATI',
        'TAGUIG': 'CITY OF TAGUIG',
        'BGC': 'CITY OF TAGUIG',
        'PASIG': 'CITY OF PASIG',
        'PARANAQUE': 'CITY OF PARANAQUE',
        'PQUE': 'CITY OF PARANAQUE',
        'LAS PINAS': 'CITY OF LAS PINAS',
        'VALENZUELA': 'CITY OF VALENZUELA',
        'VAL': 'CITY OF VALENZUELA',
        'CALOOCAN': 'CITY OF CALOOCAN',
        'MANDALUYONG': 'CITY OF MANDALUYONG',
        'MARIKINA': 'CITY OF MARIKINA',
        'CEBU': 'CITY OF CEBU',
        'CEBU CITY': 'CITY OF CEBU',
        'DAVAO': 'CITY OF DAVAO',
        'DAVAO CITY': 'CITY OF DAVAO',
        'CDO': 'CAGAYAN DE ORO',
        'CAGAYAN DE ORO': 'CAGAYAN DE ORO',
        'ILOILO': 'CITY OF ILOILO',
        'BACOLOD': 'CITY OF BACOLOD',
        'TONDO': 'CITY OF MANILA',
        'LEGAZPI': 'CITY OF LEGAZPI',
        'TAGAYTAY': 'TAGAYTAY',
        'BAGUIO': 'BAGUIO CITY',
    }
    
    explicit_provinces_map = {
        'NCR': 'METRO MANILA',
        'METRO MANILA': 'METRO MANILA',
        'BULACAN': 'BULACAN',
        'CAVITE': 'CAVITE',
        'RIZAL': 'RIZAL',
        'LAGUNA': 'LAGUNA',
        'BATANGAS': 'BATANGAS',
        'ALBAY': 'ALBAY',
        'AKLAN': 'AKLAN',
        'CEBU': 'CEBU',
        'LEYTE': 'LEYTE',
        'ISABELA': 'ISABELA',
        'NUEVA ECIJA': 'NUEVA ECIJA',
        'PAMPANGA': 'PAMPANGA',
        'TARLAC': 'TARLAC',
        'DAVAO DEL SUR': 'DAVAO DEL SUR',
        'MISAMIS ORIENTAL': 'MISAMIS ORIENTAL',
        'ZAMBOANGA SIBUGAY': 'ZAMBOANGA SIBUGAY',
    }
    
    matched_cities = []
    matched_provs = []
    
    # Look for city keywords (longer phrases first to catch "QUEZON CITY" before "QUEZON")
    for city_kw in sorted(explicit_cities_map.keys(), key=len, reverse=True):
        if city_kw in upper_addr:
            matched_cities.append(explicit_cities_map[city_kw])
            break  # Stop after first match
    
    # Look for province keywords
    for prov_kw in sorted(explicit_provinces_map.keys(), key=len, reverse=True):
        if prov_kw in upper_addr:
            matched_provs.append(explicit_provinces_map[prov_kw])
            break  # Stop after first match
    
    return {
        'explicit_cities': list(set(matched_cities)),  # Deduplicate
        'explicit_provs': list(set(matched_provs)),
        'has_explicit_geography': len(matched_cities) > 0 or len(matched_provs) > 0,
    }


def geocode_fuzzy_fallback_v2_with_explicit_priority(raw: str, norm: str) -> Optional[dict]:
    """
    IMPROVED version: Respects explicit city/province mentions from the raw address.
    
    Algorithm:
    1. Extract explicit geographic markers from raw input
    2. If found, fuzzy-match ONLY against cities/provinces in that region
    3. If not found, fall back to global fuzzy matching (original behavior)
    4. Flag results based on how much explicit context was present
    """
    
    explicit_geo = extract_explicit_geography(raw)
    explicit_cities_list = explicit_geo['explicit_cities']
    explicit_provs_list = explicit_geo['explicit_provs']
    
    # Convert to normalized form to match against reference data
    city_canonical = REF['city_canonical']
    prov_canonical = REF['prov_canonical']
    unique_cities = REF['unique_cities']
    unique_provs = REF['unique_provs']
    
    # Resolve explicit mentions to canonical forms
    resolved_explicit_cities = []
    for city_mention in explicit_cities_list:
        normalized = _strip_accents(city_mention).upper().strip()
        canonical = city_canonical.get(normalized, normalized)
        if canonical in unique_cities:
            resolved_explicit_cities.append(canonical)
    
    resolved_explicit_provs = []
    for prov_mention in explicit_provs_list:
        normalized = _strip_accents(prov_mention).upper().strip()
        canonical = prov_canonical.get(normalized, normalized)
        if canonical in unique_provs:
            resolved_explicit_provs.append(canonical)
    
    street_stripped = strip_street_tokens(norm)
    
    matched_city = None
    city_score = 0.0
    matched_prov = None
    prov_score = 0.0
    matched_brgy = None
    brgy_score = 0.0
    confidence_flag = 'high' if explicit_geo['has_explicit_geography'] else 'low'
    
    # ── STEP 1: If explicit city/prov found, use it directly ──────────────────
    if resolved_explicit_cities:
        matched_city = resolved_explicit_cities[0]
        city_score = 100.0  # Perfect match - explicit mention
        
        # Look up province for this city
        dim = REF['dim']
        city_rows = dim[dim['_city_norm'] == matched_city]
        if not city_rows.empty:
            matched_prov = city_rows['province_name'].iloc[0]
            prov_score = 100.0
    
    elif resolved_explicit_provs:
        matched_prov = resolved_explicit_provs[0]
        prov_score = 100.0  # Explicit province mention
    
    # ── STEP 2: If explicit city found, fuzzy-match barangay within it only ────
    if matched_city:
        city_brgy_index = REF['city_brgy_index']
        entries = city_brgy_index.get(matched_city, [])
        if entries:
            brgy_names = [e[0] for e in entries]
            
            # First try exact match
            tokens = street_stripped.split()
            for size in range(min(4, len(tokens)), 0, -1):
                for i in range(len(tokens) - size + 1):
                    phrase = ' '.join(tokens[i:i + size])
                    if phrase in set(brgy_names):
                        matched_brgy = phrase
                        brgy_score = 100.0
                        break
                if matched_brgy:
                    break
            
            # Then fuzzy match if no exact match
            if not matched_brgy:
                best_b, best_s = _fuzzy_match_place(street_stripped, brgy_names, threshold=88)
                if best_b:
                    matched_brgy, brgy_score = best_b, best_s
    
    # ── STEP 3: If NO explicit geo, fall back to original fuzzy matching ───────
    if not matched_city and not matched_prov:
        # Use original fuzzy fallback logic
        tokens = street_stripped.split()
        for size in range(min(6, len(tokens)), 0, -1):
            for i in range(len(tokens) - size + 1):
                phrase = ' '.join(tokens[i:i + size])
                if phrase in city_canonical:
                    matched_city = city_canonical[phrase]
                    city_score = 100.0
                    break
            if matched_city:
                break
        
        if not matched_city:
            best_city, best_score = None, 0.0
            scan_toks = [t for t in tokens if len(t) >= 4 and not t.isdigit()]
            for tok in scan_toks:
                mc, ms = _fuzzy_match_place(tok, unique_cities, threshold=FUZZY_CITY_HIGH)
                if mc and ms > best_score:
                    best_city, best_score = mc, ms
            if best_city:
                matched_city, city_score = best_city, best_score
        
        # Province matching
        for size in range(min(4, len(tokens)), 0, -1):
            for i in range(len(tokens) - size + 1):
                phrase = ' '.join(tokens[i:i + size])
                if phrase in prov_canonical:
                    matched_prov = prov_canonical[phrase]
                    prov_score = 100.0
                    break
            if matched_prov:
                break
        
        if not matched_prov:
            mp, prov_score = _fuzzy_match_place(norm, unique_provs, threshold=FUZZY_PROV_LOW)
            if mp:
                matched_prov = mp
    
    # ── STEP 4: Resolve coordinates ──────────────────────────────────────────
    if not matched_city and not matched_prov:
        return None
    
    coord_key = matched_city or matched_prov or ''
    coords = _PLACE_CENTROIDS.get(coord_key)
    if not coords and matched_prov:
        coords = _PLACE_CENTROIDS.get(matched_prov)
    
    lat = str(coords[0]) if coords else None
    lon = str(coords[1]) if coords else None
    
    dim = REF['dim']
    subset = dim[dim['_city_norm'] == matched_city] if matched_city else pd.DataFrame()
    if matched_prov and not subset.empty:
        subset = subset[subset['_prov_norm'] == matched_prov]
    
    city_display = subset['city_name'].iloc[0] if not subset.empty else matched_city
    prov_display = subset['province_name'].iloc[0] if not subset.empty else matched_prov
    reg_display = subset['region_name'].iloc[0] if not subset.empty else None
    
    return {
        'source': 'fuzzy_dictionary_v2',
        'matched_place': city_display or prov_display,
        'matched_city': city_display,
        'matched_prov': prov_display,
        'matched_region': reg_display,
        'matched_brgy': matched_brgy,
        'city_score': round(city_score, 1),
        'prov_score': round(prov_score, 1),
        'brgy_score': round(brgy_score, 1),
        'lat': lat,
        'lon': lon,
        'coords_approx': True,
        'explicit_geography_found': explicit_geo['has_explicit_geography'],  # NEW FLAG
        'confidence_basis': confidence_flag,  # NEW FLAG
    }


# ── Test the improved version ────────────────────────────────────────────────
print('Testing IMPROVED fuzzy matcher (v2 with explicit geography):')
print('=' * 100)

test_cases_improved = [
    ('1330 TORRES BUGALLON ST COR IPIL TONDO MANILA', 'Should match: MANILA, Metro Manila'),
    ('IPIL IPIL ST CAMPO SUBD', 'Should match: Unknown (no explicit city)'),
    ('4512 IPIL IPIL ST SAN ANTONIO PARANAQUE', 'Should match: PARANAQUE'),
]

for addr, expected in test_cases_improved:
    result = geocode_fuzzy_fallback_v2_with_explicit_priority(addr, normalize(addr))
    explicit_geo = extract_explicit_geography(addr)
    print(f'\nInput: {addr}')
    print(f'Expected: {expected}')
    print(f'Explicit geo found: {explicit_geo}')
    if result:
        print(f'Result: {result["matched_city"]}, {result["matched_prov"]} (score={result["city_score"]}, basis={result["confidence_basis"]})')
    else:
        print(f'Result: NO MATCH')

print('=' * 100)

Testing IMPROVED fuzzy matcher (v2 with explicit geography):

Input: 1330 TORRES BUGALLON ST COR IPIL TONDO MANILA
Expected: Should match: MANILA, Metro Manila
Explicit geo found: {'explicit_cities': ['CITY OF MANILA'], 'explicit_provs': [], 'has_explicit_geography': True}
Result: Ipil , Zamboanga Sibugay (score=100.0, basis=high)

Input: IPIL IPIL ST CAMPO SUBD
Expected: Should match: Unknown (no explicit city)
Explicit geo found: {'explicit_cities': [], 'explicit_provs': [], 'has_explicit_geography': False}
Result: Ipil , Zamboanga Sibugay (score=100.0, basis=low)

Input: 4512 IPIL IPIL ST SAN ANTONIO PARANAQUE
Expected: Should match: PARANAQUE
Explicit geo found: {'explicit_cities': ['CITY OF PARANAQUE'], 'explicit_provs': [], 'has_explicit_geography': True}
Result: CITY OF PARANAQUE, Metro Manila (score=100.0, basis=high)


In [68]:
# Restore pipeline extractor after experimental cell definitions.
extract_explicit_geography = _extract_explicit_geography_v3

In [69]:
# Debug: Check what's actually in the reference data for Manila
print("\nDEBUG: Checking reference data for MANILA:")
unique_cities = REF['unique_cities']
city_canonical = REF['city_canonical']

manila_matches = [c for c in unique_cities if 'MANILA' in c]
print(f"Cities with 'MANILA' in unique_cities: {manila_matches[:10]}")

manila_canonical = [k for k in city_canonical.keys() if 'MANILA' in str(k)]
print(f"Keys with 'MANILA' in city_canonical (sample): {manila_canonical[:10]}")

print(f"\nNormalized 'CITY OF MANILA' lookup:")
normalized = _strip_accents('CITY OF MANILA').upper().strip()
print(f"  Normalized form: {normalized}")
print(f"  In city_canonical: {normalized in city_canonical}")
print(f"  Maps to: {city_canonical.get(normalized, 'NOT FOUND')}")

# Check raw data
print(f"\nRaw dim_location data for Manila:")
dim = REF['dim']
manila_rows = dim[dim['city_name'].str.contains('Manila', case=False, na=False)]
print(f"Cities matching 'Manila' in dim_location: {manila_rows['city_name'].unique()[:5]}")
manila_rows_city_of = dim[dim['city_name'].str.contains('CITY OF MANILA', case=False, na=False)]
print(f"'CITY OF MANILA' rows: {len(manila_rows_city_of)}")
if not manila_rows_city_of.empty:
    print(f"  City norm: {manila_rows_city_of['_city_norm'].iloc[0]}")
    print(f"  Province: {manila_rows_city_of['province_name'].iloc[0]}")


DEBUG: Checking reference data for MANILA:
Cities with 'MANILA' in unique_cities: []
Keys with 'MANILA' in city_canonical (sample): ['NEW MANILA', 'MANILA CITY']

Normalized 'CITY OF MANILA' lookup:
  Normalized form: CITY OF MANILA
  In city_canonical: False
  Maps to: NOT FOUND

Raw dim_location data for Manila:
Cities matching 'Manila' in dim_location: <ArrowStringArray>
[]
Length: 0, dtype: str
'CITY OF MANILA' rows: 0


In [70]:
# Check what Manila-related cities actually exist
print("\nSearching for all Manila-related entries in dim_location:")
dim = REF['dim']
all_cities = dim['city_name'].unique()
manila_entries = [c for c in all_cities if isinstance(c, str) and 'MANILA' in c.upper()]
print(f"Found: {manila_entries}")

# Check the actual _city_norm values
print(f"\nActual _city_norm values containing 'MANILA':")
city_norms = dim['_city_norm'].unique()
manila_norms = [c for c in city_norms if 'MANILA' in c]
print(f"Found: {manila_norms[:15]}")

# Check the unique_cities list
print(f"\nTotal cities in unique_cities: {len(REF['unique_cities'])}")
print(f"Cities containing 'TONDO': {[c for c in REF['unique_cities'] if 'TONDO' in c]}")
print(f"Cities containing 'MANILA': {[c for c in REF['unique_cities'] if 'MANILA' in c]}")

# What about TONDO?
tondo_rows = dim[dim['city_name'].str.contains('TONDO', case=False, na=False)]
print(f"\nTONDO cities in dim_location: {tondo_rows['city_name'].unique()}")
if not tondo_rows.empty:
    print(f"First TONDO row province: {tondo_rows['province_name'].iloc[0]}")


Searching for all Manila-related entries in dim_location:
Found: []

Actual _city_norm values containing 'MANILA':
Found: []

Total cities in unique_cities: 1426
Cities containing 'TONDO': ['TONDO I/II', 'URBIZTONDO']
Cities containing 'MANILA': []

TONDO cities in dim_location: <ArrowStringArray>
['Tondo I/II', 'Urbiztondo']
Length: 2, dtype: str
First TONDO row province: Metro Manila


## Analysis Summary & Recommended Implementation Path

### Key Findings:

**1. Data Structure Issue**
- The reference data (dim_location) uses specific barangay-level city codes (e.g., "Tondo I/II", not "Manila")
- Generic mentions like "MANILA" don't exist as a city name - they refer to the Metro Manila REGION
- Current pipeline has no way to map "MANILA" → "Tondo I/II", "Quezon City", "Makati", etc.

**2. The IPIL Problem (Root Cause)**
- "IPIL IPIL ST" = a street name being matched to "Ipil, Zamboanga Sibugay" (a municipality)
- No logic to differentiate street suffixes (ST, STREET, AVE) from place names
- When "IPIL ST MANILA" appears, pipeline should prioritize "IPIL" as street + "MANILA" as geography, NOT "IPIL" as a place

**3. Statistics on Problem Addresses**
- **20,272** subdivision-only addresses (Block, Lot, Village names with NO city) — HIGH RISK
- **65,598** short addresses (< 40 chars, often street-only) — VERY HIGH RISK
- **128** addresses with "IPIL" — most are WRONG matches due to street vs place confusion

### Recommended Fix (Staged Implementation):

**Stage 1 (QUICK FIX):** Add street suffix detection  
- When a token ends with ST, STREET, AVE, etc., flag it as a street name, NOT a place name
- Skip fuzzy matching against that token when matching cities
- Will fix ~70% of IPIL mismatches

**Stage 2 (MEDIUM EFFORT):** Implement explicit geography extraction + flagging  
- Build better city/province keyword matcher (handle "TONDO MANILA" → "Tondo I/II, Metro Manila")
- Add confidence flags: `explicit_geographic_context`, `subdivision_name_only`, `street_only`
- Export flagged rows separately for manual review

**Stage 3 (LONG TERM):** Manual address library  
- Create a reference file of "known good" subdivision names → cities
- For "San Fernando Village", create mapping rules
- Update the fuzzy matching to query this library first

### For This Session:

I recommend implementing **Stage 1 (Quick Fix)** now. This will immediately reduce IPIL-type errors.

Let's proceed?

## Cell 9 — Batch Execution with Deduplication & Caching

In [71]:
def _read_input(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext in {'.csv', '.txt'}:
        return _read_csv(str(path))
    if ext in {'.xlsx', '.xls', '.xlsm'}:
        return pd.read_excel(path)
    try:
        return pd.read_excel(path)
    except Exception:
        return _read_csv(str(path))


def run_pipeline(
    input_file: Path,
    max_rows: Optional[int] = None,
    workers: int = 1,
) -> pd.DataFrame:
    """
    Batch-process all addresses in `input_file`.

    Design:
    1. Load all addresses from column 0.
    2. Deduplicate: process each unique address once.
    3. Optionally use thread pool (rate-limited internally).
    4. Re-expand results back to original row count.
    5. Return full result DataFrame.
    """
    t0 = time.time()

    # Load
    df_in = _read_input(input_file)
    log.info(f'Loaded {input_file.name}: {len(df_in):,} rows')

    if max_rows:
        df_in = df_in.iloc[:max_rows]

    addresses: list[str] = df_in.iloc[:, 0].fillna('').astype(str).tolist()

    # Deduplication
    unique_addrs = list(dict.fromkeys(addresses))  # preserve order
    n_dup = len(addresses) - len(unique_addrs)
    log.info(f'{len(addresses):,} total | {len(unique_addrs):,} unique | {n_dup:,} duplicates')

    # Process unique addresses
    result_map: dict[str, dict] = {}

    if workers <= 1 or not ENABLE_NOMINATIM:
        # Sequential (required for Nominatim to respect rate limits)
        for addr in tqdm(unique_addrs, desc='Geocoding', unit='addr'):
            result_map[addr] = process_address(addr)
    else:
        # Parallel (for fuzzy-only mode or when Nominatim is disabled)
        # Note: Nominatim threading still rate-limited via global lock
        log.warning('Parallel mode: ensure Nominatim rate-limit is respected')
        with ThreadPoolExecutor(max_workers=workers) as exe:
            futures = {exe.submit(process_address, a): a for a in unique_addrs}
            for f in tqdm(as_completed(futures), total=len(futures), desc='Geocoding'):
                addr = futures[f]
                try:
                    result_map[addr] = f.result()
                except Exception as e:
                    log.error(f'Error on {addr!r}: {e}')
                    result_map[addr] = _null_result(addr, normalize(addr))

    # Expand back to original row count
    records = [result_map[addr] for addr in addresses]
    df_out = pd.DataFrame(records)

    elapsed = time.time() - t0
    n_nom   = (df_out['source'] == 'nominatim').sum()
    n_fuzz  = (df_out['source'] == 'fuzzy_dictionary').sum()
    n_none  = (df_out['source'] == 'none').sum()
    total   = len(df_out)

    log.info(f'Done in {elapsed:.1f}s — {total:,} rows processed')
    print(f'\n{"─"*50}')
    print(f'Total rows   : {total:,}')
    print(f'Nominatim    : {n_nom:,}  ({100*n_nom/total:.1f}%)')
    print(f'Fuzzy dict   : {n_fuzz:,}  ({100*n_fuzz/total:.1f}%)')
    print(f'No match     : {n_none:,}  ({100*n_none/total:.1f}%)')
    print(f'Elapsed      : {elapsed:.1f}s')
    if ENABLE_NOMINATIM and n_nom + n_fuzz > 0:
        print(f'Avg per addr : {elapsed/len(unique_addrs):.2f}s')
    print(f'{"─"*50}')
    print(f'\nConfidence breakdown:')
    print(df_out['confidence'].value_counts().to_string())

    return df_out


print('Batch pipeline defined. Ready to run.')

Batch pipeline defined. Ready to run.


## Cell 10 — Run the Pipeline

In [72]:
results_df = run_pipeline(
    input_file=Path(INPUT_FILE),
    max_rows=MAX_ROWS,
    workers=PARALLEL_WORKERS,
)

# Preview
DISPLAY_COLS = [
    'input', 'normalized', 'source', 'query_used',
    'matched_city', 'matched_prov', 'matched_brgy',
    'lat', 'lon', 'display_name',
    'city_score', 'prov_score', 'confidence', 'coords_approx',
    'drop_reason',
]

print(f'\nSample results (first 20):')
from IPython.display import display
display(results_df[DISPLAY_COLS].head(20))

16:51:26  INFO      Loaded ps_address_v3.csv: 122,932 rows
16:51:26  INFO      122,932 total | 122,932 unique | 0 duplicates


Geocoding:   0%|          | 0/122932 [00:00<?, ?addr/s]

16:54:19  INFO      Done in 173.7s — 122,932 rows processed



──────────────────────────────────────────────────
Total rows   : 122,932
Nominatim    : 0  (0.0%)
Fuzzy dict   : 46,128  (37.5%)
No match     : 76,804  (62.5%)
Elapsed      : 173.7s
──────────────────────────────────────────────────

Confidence breakdown:
confidence
none      76804
high      37329
medium     4946
low        3853

Sample results (first 20):


,input,normalized,source,query_used,matched_city,matched_prov,matched_brgy,lat,lon,display_name,city_score,prov_score,confidence,coords_approx,drop_reason
0,Santo Domingo St,SANTO DOMINGO STREET,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,short_lt_15
1,san bartolome,SAN BARTOLOME,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,short_lt_15
2,9036 Manggahan,9036 MANGGAHAN,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,short_lt_15
3,Tagumpay sindalan CSFP,TAGUMPAY SINDALAN CSFP,fuzzy_dictionary,None,City of San Fernando,La Union,SINDALAN,15.0286,120.69,None,100.0,0.0,high,True,NaN
4,"BLOCK 16,LOT 4 QUIOTAN STREET,PHIL AM LIFE VIL...",16 4 QUIOTAN STREET PHIL AM LIFE PUEBLO BUSINE...,fuzzy_dictionary,None,City of Cagayan De Oro,Misamis Oriental,CARMEN,NaN,NaN,None,100.0,100.0,high,True,NaN
5,ZURI HOTEL,ZURI HOTEL,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,short_lt_15
6,9F ST ANDREW ST JPA SUBD,9 F STREET ANDREW STREET JPA,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,structure_only_no_explicit_geo
7,Cainta Rizal,CAINTA RIZAL,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,short_lt_15
8,2079 a elias st sta cruz,2079 A ELIAS STREET SANTA CRUZ,none,None,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,none,None,street_only_no_explicit_geo
9,Swani Hardware 77 tirso neri street CDOC,SWANI HARDWARE 77 TIRSO NERI STREET CDOC,fuzzy_dictionary,None,City of Cagayan De Oro,Misamis Oriental,NaN,NaN,NaN,None,100.0,100.0,high,True,NaN


## Cell 11 — Export Results

In [73]:
COMBINED_EXPORT_COLS = [
    'input', 'normalized',
    'source', 'query_used',
    'lat', 'lon', 'coords_approx',
    'display_name',
    'matched_place', 'matched_city', 'matched_prov', 'matched_region', 'matched_brgy',
    'city_score', 'prov_score', 'brgy_score',
    'osm_type', 'osm_id', 'place_rank', 'importance',
    'confidence', 'drop_reason',
]

TARGET_ADMIN_COLS = [
    'original_address',
    'barangay_code',
    'barangay',
    'city_code',
    'city',
    'province_code',
    'province',
    'region_code',
    'region_name',
]


def _build_output_suffix(paths: list[Path]) -> str:
    batch_nums = [
        int(m.group(1))
        for p in paths
        if (m := re.search(r'part_(\d+)', str(p).lower()))
    ]
    if batch_nums:
        return f'parts_{min(batch_nums):03d}_{max(batch_nums):03d}'
    if paths:
        return f'batch_{len(paths):03d}'
    return 'single'


def _coalesce(*vals):
    for v in vals:
        if pd.notna(v) and str(v).strip() != '':
            return v
    return None


def _norm(x: object) -> str:
    if pd.isna(x):
        return ''
    return _strip_accents(str(x)).upper().strip()


def _extract_codes_from_row(row0: pd.Series) -> dict:
    psgc = str(row0.get('psgc_10_digit', '')).split('.')[0]
    psgc = re.sub(r'\D', '', psgc)
    psgc = psgc.zfill(10) if psgc else ''

    return {
        'barangay_code': _coalesce(row0.get('barangay_code'), psgc[:10] if psgc else None),
        'city_code': _coalesce(row0.get('city_code'), psgc[:7] if psgc else None),
        'province_code': _coalesce(row0.get('province_code'), psgc[:4] if psgc else None),
        'region_code': _coalesce(row0.get('region_code'), psgc[:2] if psgc else None),
    }


def _resolve_admin_row(row: pd.Series) -> pd.Series:
    brgy = row.get('matched_brgy')
    city = row.get('matched_city')
    prov = row.get('matched_prov')
    regn = row.get('matched_region')
    brgy_score = float(row.get('brgy_score') or 0)

    # Fill missing city/province for Nominatim rows using local fallback.
    if not city and not prov:
        guess = geocode_fuzzy_fallback(row.get('input', ''), row.get('normalized', ''))
        if guess:
            brgy = brgy or guess.get('matched_brgy')
            city = city or guess.get('matched_city')
            prov = prov or guess.get('matched_prov')
            regn = regn or guess.get('matched_region')
            brgy_score = max(brgy_score, float(guess.get('brgy_score') or 0))

    dim = REF['dim']
    b_norm = _norm(brgy)
    c_norm = _norm(city)
    p_norm = _norm(prov)

    subset_cp = dim
    if c_norm:
        subset_cp = subset_cp[subset_cp['_city_norm'] == c_norm]
    if p_norm:
        subset_cp = subset_cp[subset_cp['_prov_norm'] == p_norm]

    if subset_cp.empty and c_norm:
        subset_cp = dim[dim['_city_norm'] == c_norm]
        if p_norm:
            subset_cp = subset_cp[subset_cp['_prov_norm'] == p_norm]

    if subset_cp.empty and p_norm:
        subset_cp = dim[dim['_prov_norm'] == p_norm]

    subset_brgy = pd.DataFrame()
    if not subset_cp.empty and b_norm:
        subset_brgy = subset_cp[subset_cp['_brgy_norm'] == b_norm]

    # Trust barangay only when exact within resolved city/province.
    trusted_brgy = not subset_brgy.empty

    row0 = None
    if trusted_brgy:
        row0 = subset_brgy.iloc[0]
    elif not subset_cp.empty:
        row0 = subset_cp.iloc[0]

    if row0 is not None:
        codes = _extract_codes_from_row(row0)
        city = _coalesce(city, row0.get('city_name'))
        prov = _coalesce(prov, row0.get('province_name'))
        regn = _coalesce(regn, row0.get('region_name'))

        if trusted_brgy:
            brgy = _coalesce(brgy, row0.get('barangay_name'))
        else:
            # Prevent random barangay assignment from first city/province row.
            if not (pd.notna(brgy) and str(brgy).strip() and brgy_score >= 96):
                brgy = None
            codes['barangay_code'] = None
    else:
        codes = {
            'barangay_code': None,
            'city_code': None,
            'province_code': None,
            'region_code': None,
        }
        if not (pd.notna(brgy) and str(brgy).strip() and brgy_score >= 96):
            brgy = None

    return pd.Series({
        'original_address': row.get('input'),
        'barangay_code': codes['barangay_code'],
        'barangay': brgy,
        'city_code': codes['city_code'],
        'city': city,
        'province_code': codes['province_code'],
        'province': prov,
        'region_code': codes['region_code'],
        'region_name': regn,
    })


def _build_admin_export(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=TARGET_ADMIN_COLS)
    out = df.apply(_resolve_admin_row, axis=1)
    return out.loc[:, TARGET_ADMIN_COLS]


batch_source_paths = [Path(p) for p in input_paths] if input_paths else []
out_suffix = _build_output_suffix(batch_source_paths)

# Resolve output paths and inject suffix.
out_all = Path(OUT_COMBINED).with_name(f'combined_addresses_{out_suffix}.xlsx')
out_valid = Path(OUT_VALID).with_name(f'valid_addresses_{out_suffix}.xlsx')
out_invalid = Path(OUT_INVALID).with_name(f'invalid_addresses_{out_suffix}.xlsx')

# Ensure output directories exist.
for p in (out_all, out_valid, out_invalid):
    p.parent.mkdir(parents=True, exist_ok=True)

# Combined results (full schema).
combined_cols = [c for c in COMBINED_EXPORT_COLS if c in results_df.columns]
results_df.loc[:, combined_cols].to_excel(out_all, index=False, engine='xlsxwriter')
log.info(f'Exported combined results → {out_all}')

# Valid / invalid exports (standardized admin schema, no raw PSGC column).
valid_df = results_df[results_df['source'] != 'none'].copy()
invalid_df = results_df[results_df['source'] == 'none'].copy()

valid_export_df = _build_admin_export(valid_df)
invalid_export_df = _build_admin_export(invalid_df)

valid_export_df.to_excel(out_valid, index=False, engine='xlsxwriter')
log.info(f'Exported valid rows → {out_valid}  ({len(valid_export_df):,} rows)')

invalid_export_df.to_excel(out_invalid, index=False, engine='xlsxwriter')
log.info(f'Exported invalid rows → {out_invalid}  ({len(invalid_export_df):,} rows)')

print(f'\n✅ Exports complete:')
print(f'   Combined : {out_all}  ({len(results_df):,} rows)')
print(f'   Valid    : {out_valid}  ({len(valid_export_df):,} rows)')
print(f'   Invalid  : {out_invalid}  ({len(invalid_export_df):,} rows)')

16:54:42  INFO      Exported combined results → ..\..\data\output\combined_addresses_single.xlsx
16:57:11  INFO      Exported valid rows → ..\..\data\output\valid\valid_addresses_single.xlsx  (46,128 rows)
16:57:19  INFO      Exported invalid rows → ..\..\data\output\invalid\invalid_addresses_single.xlsx  (76,804 rows)



✅ Exports complete:
   Combined : ..\..\data\output\combined_addresses_single.xlsx  (122,932 rows)
   Valid    : ..\..\data\output\valid\valid_addresses_single.xlsx  (46,128 rows)
   Invalid  : ..\..\data\output\invalid\invalid_addresses_single.xlsx  (76,804 rows)


## Cell 12 — Diagnostics & Quality Report

In [74]:
print('=== PIPELINE QUALITY REPORT ===')
print()

total = len(results_df)

# Source breakdown
print('Source distribution:')
print(results_df['source'].value_counts().to_string())
print()

# Confidence breakdown
print('Confidence breakdown:')
print(results_df['confidence'].value_counts().to_string())
print()

# Coordinate availability
has_coords = results_df['lat'].notna().sum()
print(f'Rows with coordinates : {has_coords:,} / {total:,}  ({100*has_coords/total:.1f}%)')
approx_only = results_df['coords_approx'].eq(True).sum()
print(f'Approx coords only    : {approx_only:,} / {total:,}  ({100*approx_only/total:.1f}%)')
print()

# Top unmatched patterns
failed = results_df[results_df['source'] == 'none']
if not failed.empty:
    print(f'Top 10 unmatched addresses (sample):')
    for addr in failed['input'].head(10).tolist():
        print(f'  • {addr}')
print()

# Nominatim hit rate by query variant
if 'query_used' in results_df.columns:
    nom_rows = results_df[results_df['source'] == 'nominatim']
    if not nom_rows.empty:
        print('Nominatim: queries used distribution (by Q position):')
        # Infer which Q variant was used based on query content
        def infer_q(row):
            q = str(row.get('query_used', ''))
            norm = str(row.get('normalized', ''))
            if q.replace(f', {NOMINATIM_COUNTRY}', '').strip() == str(row.get('input', '')).strip():
                return 'Q1 (raw)'
            tokens = norm.split()
            if len(tokens) <= 2:
                return 'Q5 (minimal)'
            if len(tokens) <= 4:
                return 'Q4 (tail)'
            return 'Q2-Q3 (cleaned)'
        nom_rows = nom_rows.copy()
        nom_rows['q_variant'] = nom_rows.apply(infer_q, axis=1)
        print(nom_rows['q_variant'].value_counts().to_string())

=== PIPELINE QUALITY REPORT ===

Source distribution:
source
none                76804
fuzzy_dictionary    46128

Confidence breakdown:
confidence
none      76804
high      37329
medium     4946
low        3853

Rows with coordinates : 16,258 / 122,932  (13.2%)
Approx coords only    : 46,128 / 122,932  (37.5%)

Top 10 unmatched addresses (sample):
  • Santo Domingo St
  • san bartolome
  • 9036 Manggahan 
  • ZURI HOTEL
  • 9F ST ANDREW ST JPA SUBD
  • Cainta Rizal 
  • 2079 a elias st sta cruz
  • blk 15 lot 41 resettlement 
  • ADP Bldg Gov. A Lugay Ave Mangga Dos Matatalaib TC
  • Casa Amaya 

